# Test — Final Integration (Day 9)

In [0]:
# Test — Final Integration (Day 9)
# Different in kind from every other test file in this project: those each
# check ONE day's output in isolation. This checks that Bronze -> Silver ->
# Gold hold together as a single connected system, after end_to_end_pipeline_job
# runs the whole thing. Row counts must reconcile ACROSS layers, not just
# within one table.

import unittest

CATALOG = "vstone_catalog"


class FinalIntegrationTests(unittest.TestCase):

    # ── Cross-layer row-count reconciliation ──────────────────────────────

    def test_streets_bronze_to_gold_reconciles(self):
        """
        All 4 Bronze street sources -> streets_silver (minus anything
        legitimately quarantined) -> fact_street_readings. This is the
        single most important check in the whole project: it's the thing
        that would have caught the location=7 orphan bug and the cars.csv
        grain bug immediately, instead of discovering them one test run
        at a time across several days.
        """
        bronze_total = sum(
            spark.table(f"{CATALOG}.bronze.{t}").count()
            for t in ["streets_csv_copyinto", "streets_autoloader", "streets_dlt", "streets_xml"]
        )
        silver_valid = spark.table(f"{CATALOG}.silver.streets_silver").count()
        silver_quarantined = spark.table(f"{CATALOG}.silver.streets_silver_quarantine").count()
        gold_fact = spark.table(f"{CATALOG}.gold.fact_street_readings").count()

        self.assertEqual(
            silver_valid + silver_quarantined, bronze_total,
            f"Bronze->Silver gap for streets: bronze={bronze_total:,}, "
            f"silver valid+quarantine={silver_valid + silver_quarantined:,}"
        )
        self.assertEqual(
            gold_fact, silver_valid,
            f"Silver->Gold gap for streets: silver={silver_valid:,}, gold={gold_fact:,}"
        )

    def test_cars_bronze_to_gold_reconciles(self):
        bronze_total = spark.table(f"{CATALOG}.bronze.cars_dlt").count()
        silver_valid = spark.table(f"{CATALOG}.silver.cars_silver").count()
        silver_quarantined = spark.table(f"{CATALOG}.silver.cars_silver_quarantine").count()
        gold_fact = spark.table(f"{CATALOG}.gold.fact_traffic_counts").count()

        # Bronze->Silver gap here is EXPECTED to be > quarantine alone —
        # dropDuplicates on the confirmed (location, reading_ts, reading_id)
        # grain removes true duplicates, which are not quarantined rows.
        # What must NOT happen: silver+quarantine < bronze by more than the
        # confirmed duplicate count would explain (that would mean rows are
        # vanishing without being quarantined OR deduplicated).
        accounted_for = silver_valid + silver_quarantined
        self.assertLessEqual(
            accounted_for, bronze_total,
            f"cars: silver+quarantine ({accounted_for:,}) exceeds bronze ({bronze_total:,}) — impossible, investigate."
        )
        self.assertEqual(
            gold_fact, silver_valid,
            f"Silver->Gold gap for cars: silver={silver_valid:,}, gold={gold_fact:,}"
        )

    # ── Dimension referential integrity, checked holistically ─────────────

    def test_no_orphan_facts_against_any_active_dimension(self):
        """
        The general-purpose version of the location=7 bug check — run once
        against every fact/dimension pair, not just the one that already
        bit us.
        """
        checks = [
            ("fact_street_readings", "street_id", "dim_street", "street_id"),
            ("fact_traffic_counts", "location", "dim_node_location", "location"),
        ]
        for fact_t, fact_col, dim_t, dim_col in checks:
            with self.subTest(fact=fact_t, dim=dim_t):
                fact = spark.table(f"{CATALOG}.gold.{fact_t}")
                dim = spark.table(f"{CATALOG}.gold.{dim_t}").filter("__END_AT IS NULL").select(dim_col)
                orphans = fact.join(dim, fact[fact_col] == dim[dim_col], "left_anti").count()
                self.assertEqual(orphans, 0, f"{orphans} rows in {fact_t} have no matching active row in {dim_t}.")

    # ── Business rule survival across layers ──────────────────────────────

    def test_raining_clip_rule_present_in_gold_not_silver(self):
        """Confirms the Day 4/5 day-boundary is still intact after every layer runs together."""
        silver_out_of_range = spark.table(f"{CATALOG}.silver.streets_silver") \
            .filter("raining < 0 OR raining > 100").count()
        gold_out_of_range = spark.table(f"{CATALOG}.gold.fact_street_readings") \
            .filter("raining_clipped < 0 OR raining_clipped > 100").count()
        self.assertGreater(silver_out_of_range, 0, "Silver should still have unclipped out-of-range values.")
        self.assertEqual(gold_out_of_range, 0, "Gold's raining_clipped must never be out of [0,100].")

    # ── Every quarantine table has a real, non-empty reason for every row ──

    def test_all_quarantine_tables_have_reasons(self):
        quarantine_tables = ["cars_silver_quarantine", "telegram_silver_quarantine",
                              "streets_silver_quarantine", "node_locations_silver_quarantine",
                              "streets_list_silver_quarantine"]
        for t in quarantine_tables:
            with self.subTest(table=t):
                df = spark.table(f"{CATALOG}.silver.{t}")
                if df.count() == 0:
                    continue
                missing_reason = df.filter("quarantine_reason IS NULL OR quarantine_reason = ''").count()
                self.assertEqual(missing_reason, 0, f"{t} has rows with no quarantine_reason.")

    # ── Governance views resolve without error against live Gold tables ───

    def test_governance_views_resolve_against_current_gold_state(self):
        for view in ["security.rls_fact_street_readings", "security.rls_monthly_street_trend",
                     "security.cls_dim_street", "security.cls_fact_citizen_reports"]:
            with self.subTest(view=view):
                try:
                    spark.table(f"{CATALOG}.{view}").count()
                except Exception as e:
                    self.fail(f"{view} failed to resolve: {e}")


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(FinalIntegrationTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Final integration tests FAILED — see output above.")
